# Revenue by Cohort

Question: how does revenue vary by signup cohort and order month?


## Profile Summary

Inputs are local Parquet customer and order extracts. Profile before analysis: row counts, join keys, unmatched orders, date ranges, and revenue ranges are checked before cohort aggregation.


In [ ]:
from pathlib import Path
import pandas as pd

base = Path('dogfood/com47')
customers = pd.read_parquet(base / 'data/customers.parquet')
orders = pd.concat([
    pd.read_parquet(base / 'data/orders_q1.parquet'),
    pd.read_parquet(base / 'data/orders_q2.parquet'),
], ignore_index=True)
customers.shape, orders.shape


In [ ]:
customers['signup_month'] = pd.to_datetime(customers['signup_date']).dt.to_period('M').astype(str)
orders['order_month'] = pd.to_datetime(orders['order_date']).dt.to_period('M').astype(str)
joined = orders.merge(customers[['customer_id', 'signup_month', 'region']], on='customer_id', how='left', indicator=True)
joined['_merge'].value_counts()


## Findings and Caveats

- This notebook is a reviewable exploration shell, not production logic.
- The rerunnable script in `dogfood/com47/scripts/build_cohort_report.py` is the source for committed outputs.
- One unmatched order is expected from the fixture and should be called out before interpreting cohort revenue.
